In [ ]:
import os
import pandas as pd
pd.set_option('display.max_columns', None)
from gpt_utils import *

%load_ext autoreload
%autoreload 2

In [2]:
!source setup_env.sh

  Using cached tornado-6.4-cp38-abi3-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (435 kB)
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.1
    Uninstalling tornado-6.5.1:
      Successfully uninstalled tornado-6.5.1

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


### GoEmotions Classification

In [3]:
goemotions_df = get_dataset('data/goemotions_test_50.csv')
goemotions_df = SanitizeGoEmotionsDataFrame(goemotions_df)
goemotions_samples = goemotions_df.sample(10, random_state=10).reset_index(); goemotions_samples

,index,id,text,label,human_label
0,37,ee2k9xt,Hey that's a thought! Maybe we need [NAME] to ...,27,neutral
1,23,edr2ac7,"Well, there's cubs and otters too.",27,neutral
2,44,ef0puf0,A surprise turn of events! I'm so glad you hea...,"17,26","joy,surprise"
3,42,ee1veao,I think the fan base is mostly past that at th...,27,neutral
4,47,efh4i9q,This is honestly the cherry on top of the cake...,1,amusement
5,20,eeyh5zd,"I know you're joking, but there are people her...",1,amusement
6,3,eelgwd1,"I didn't know that, thank you for teaching me ...",15,gratitude
7,30,edlt8ec,NJ has zero of their own picks from the 2010 d...,27,neutral
8,7,ef0ec3b,100%! Congrats on your job too!,15,gratitude
9,6,efdbh17,You’re welcome,15,gratitude


In [4]:
goemotions_classification_4 = GetGptAnnotationWithRetriesForGoEmotionsClassification(goemotions_samples, DEPLOYMENT_NAME_4, max_retries=5, verbose=True)

OpenAIError: Missing credentials. Please pass one of `api_key`, `azure_ad_token`, `azure_ad_token_provider`, or the `AZURE_OPENAI_API_KEY` or `AZURE_OPENAI_AD_TOKEN` environment variables.

In [11]:
goemotions_classification_35 = GetGptAnnotationWithRetriesForGoEmotionsClassification(goemotions_samples, DEPLOYMENT_NAME_35, max_retries=5, verbose=True)

Sample#0: Text is 'I live here! Apparently I need to be keeping my eyes open.'
	Attempt#1: raw  <neutral> sanitized to <neutral> (PASSED)
Sample#1: Text is 'What GDP growth? The one that barely broke 3-4% since 1984 and has dwindled to 1-2% to 0 to negative 1%?'
	Attempt#1: raw  <disappointment, annoyance> sanitized to <annoyance,disappointment> (PASSED)
Sample#2: Text is 'I have trophy jeans and a trophy bra! Although RIP my boobs because they were the one thing I liked about my body back then :)'
	Attempt#1: raw  <amusement, disapproval> sanitized to <amusement,disapproval> (PASSED)
Sample#3: Text is '[NAME] boxing gym in Ross is super inviting to everyone! It's a legit boxing club that is also friendly to folks just looking for a workout.'
	Attempt#1: raw  <admiration, approval> sanitized to <admiration,approval> (PASSED)
Sample#4: Text is 'So can we all collectively accept that [NAME] has a high placement for this week? Okay thank you'
	Attempt#1: raw  <approval, neutral> sanitized

In [12]:
goemotions_samples['classification_4'] = goemotions_classification_4
goemotions_samples['classification_35'] = goemotions_classification_35
goemotions_samples

,index,id,text,human_label,classification_4,classification_35
0,330,ef8avyw,I live here! Apparently I need to be keeping m...,neutral,surprise,neutral
1,1832,ef6xdiq,What GDP growth? The one that barely broke 3-4...,curiosity,"annoyance,disappointment","annoyance,disappointment"
2,3209,ee5i0j2,I have trophy jeans and a trophy bra! Although...,"disappointment,neutral","amusement,sadness","amusement,disapproval"
3,3381,edpw2z1,[NAME] boxing gym in Ross is super inviting to...,admiration,"admiration,approval","admiration,approval"
4,3395,edwe3w0,So can we all collectively accept that [NAME] ...,gratitude,"approval,optimism","approval,neutral"
5,2030,ee8kh9s,Yep. He also said it’d happen on a Manbij patrol,"approval,neutral",neutral,neutral
6,3575,eeabdmg,"Laughing at ""indoctrinated by traditional valu...","annoyance,optimism","amusement,disapproval","amusement,annoyance"
7,3799,eevggs1,"Ok, thats my thinking too. If [NAME] really on...","annoyance,disappointment","annoyance,disappointment","annoyance,disapproval,neutral"
8,1978,eczkjma,Are you daft ... ? That is and always has been...,anger,"anger,annoyance,disapproval",anger
9,3475,edkh28y,What kind of behaviour is considered unattract...,confusion,curiosity,"confusion,curiosity,neutral"


In [19]:
goemotions_samples.to_csv('test.csv')

### Emobank Regression

In [13]:
emobank_df = get_dataset('data/emobank_reader_test.csv')

In [14]:
emobank_df['weight'] = abs(emobank_df['V'] - 3) + abs(emobank_df['A'] - 3)
emobank_samples = emobank_df.sample(10, weights = emobank_df.weight, random_state=SEED); emobank_samples

,Unnamed: 0,id,text,V,A,weight
392,392,Nathans_Bylichka_47387_47415,“That was very kind of her.”,4.0,3.0,1.0
949,949,sucker_2143_2226,I read that article in the Times and IMMEDIATE...,2.6,3.8,1.2
734,734,captured_moments_19437_19659,"The next evening, she arrived with a stack of ...",3.4,2.8,0.6
570,570,The_Black_Willow_6754_6832,There simply is no audience for fairy tales – ...,2.4,3.0,0.6
172,172,Bartok_11929_12054,"As Example 10 shows, the chords are more-or-le...",3.2,2.8,0.4
57,57,116CUL034_979_1118,A Providers Council and a Youth Coalition are ...,3.8,3.2,1.0
865,865,hotel-california_18330_18380,"I guess we all figured, he knew what he was do...",3.2,3.2,0.4
574,574,The_Black_Willow_8763_8866,The reply came from above him and Allan turned...,3.2,3.0,0.2
705,705,audubon2_2158_2253,I’ve had a look at the magazine’s upcoming sub...,3.2,3.2,0.4
16,16,110CYL200_1611_1668,"Some are blind, deaf or have other physical di...",2.0,3.0,1.0


In [15]:
emobank_regression_4 = GetGptAnnotationWithRetriesForEmoBankRegression(emobank_samples, DEPLOYMENT_NAME_4, max_retries=2, verbose=True)

Sample#392: Text is '“That was very kind of her.”'
	Attempt#1: raw  <5> sanitized to <5> (PASSED)
Sample#949: Text is 'I read that article in the Times and IMMEDIATELY thought it was going to be a hoax.'
	Attempt#1: raw  <2> sanitized to <2> (PASSED)
Sample#734: Text is 'The next evening, she arrived with a stack of glistening stopboxes containing sushi, sashimi, oysters in their shells, and Terran vegetables fresh plucked from their hydroponic beds.'
	Attempt#1: raw  <5> sanitized to <5> (PASSED)
Sample#570: Text is 'There simply is no audience for fairy tales – no literate audience, at least."'
	Attempt#1: raw  <2> sanitized to <2> (PASSED)
Sample#172: Text is 'As Example 10 shows, the chords are more-or-less in duple meter (4/8 and 2/8) with one “hiccup” occurring in the 5/8 measure.'
	Attempt#1: raw  <3> sanitized to <3> (PASSED)
Sample#57: Text is 'A Providers Council and a Youth Coalition are building relationships that will hopefully bear great fruit in the years to come.'
	Atte

In [16]:
emobank_regression_35 = GetGptAnnotationWithRetriesForEmoBankRegression(emobank_samples, DEPLOYMENT_NAME_35, max_retries=2, verbose=True)

Sample#392: Text is '“That was very kind of her.”'
	Attempt#1: raw  <5> sanitized to <5> (PASSED)
Sample#949: Text is 'I read that article in the Times and IMMEDIATELY thought it was going to be a hoax.'
	Attempt#1: raw  <2> sanitized to <2> (PASSED)
Sample#734: Text is 'The next evening, she arrived with a stack of glistening stopboxes containing sushi, sashimi, oysters in their shells, and Terran vegetables fresh plucked from their hydroponic beds.'
	Attempt#1: raw  <4> sanitized to <4> (PASSED)
Sample#570: Text is 'There simply is no audience for fairy tales – no literate audience, at least."'
	Attempt#1: raw  <2> sanitized to <2> (PASSED)
Sample#172: Text is 'As Example 10 shows, the chords are more-or-less in duple meter (4/8 and 2/8) with one “hiccup” occurring in the 5/8 measure.'
	Attempt#1: raw  <3> sanitized to <3> (PASSED)
Sample#57: Text is 'A Providers Council and a Youth Coalition are building relationships that will hopefully bear great fruit in the years to come.'
	Atte

In [17]:
emobank_samples['regression_4'] = emobank_regression_4
emobank_samples['regression_35'] = emobank_regression_35
emobank_samples

,Unnamed: 0,id,text,V,A,weight,regression_4,regression_35
392,392,Nathans_Bylichka_47387_47415,“That was very kind of her.”,4.0,3.0,1.0,5,5
949,949,sucker_2143_2226,I read that article in the Times and IMMEDIATE...,2.6,3.8,1.2,2,2
734,734,captured_moments_19437_19659,"The next evening, she arrived with a stack of ...",3.4,2.8,0.6,5,4
570,570,The_Black_Willow_6754_6832,There simply is no audience for fairy tales – ...,2.4,3.0,0.6,2,2
172,172,Bartok_11929_12054,"As Example 10 shows, the chords are more-or-le...",3.2,2.8,0.4,3,3
57,57,116CUL034_979_1118,A Providers Council and a Youth Coalition are ...,3.8,3.2,1.0,5,4
865,865,hotel-california_18330_18380,"I guess we all figured, he knew what he was do...",3.2,3.2,0.4,3,3
574,574,The_Black_Willow_8763_8866,The reply came from above him and Allan turned...,3.2,3.0,0.2,3,3
705,705,audubon2_2158_2253,I’ve had a look at the magazine’s upcoming sub...,3.2,3.2,0.4,5,5
16,16,110CYL200_1611_1668,"Some are blind, deaf or have other physical di...",2.0,3.0,1.0,3,2
